# 02 — Feature Engineering & Preprocessing
**Depends on:** 01_eda.ipynb (understand which columns to drop)

Outputs: `data/processed/train.parquet`, `data/processed/test.parquet`

## 1. Setup

In [2]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from imblearn.combine import SMOTETomek
import warnings

from src.features import engineer_features

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
print('imports ok')

imports ok


## 2. Load Raw Data

In [3]:
DATA = '../data/'

train_txn = pd.read_csv(DATA + 'train_transaction.csv')
train_id  = pd.read_csv(DATA + 'train_identity.csv')
test_txn  = pd.read_csv(DATA + 'test_transaction.csv')
test_id   = pd.read_csv(DATA + 'test_identity.csv')

train = train_txn.merge(train_id, on='TransactionID', how='left')
test  = test_txn.merge(test_id,   on='TransactionID', how='left')

for df in [train, test]:
    f64 = df.select_dtypes('float64').columns
    df[f64] = df[f64].astype('float32')

print(f'Train: {train.shape}  Test: {test.shape}')

Train: (590540, 434)  Test: (506691, 433)


## 3. Drop High-Missing Columns (>95%)

In [4]:
drop_cols = train.columns[train.isnull().mean() > 0.95].tolist()
print(f'Dropping {len(drop_cols)} columns with >95% missing')

train.drop(columns=drop_cols, inplace=True)
test.drop(columns=[c for c in drop_cols if c in test.columns], inplace=True)

print(f'After drop — Train: {train.shape}  Test: {test.shape}')

Dropping 9 columns with >95% missing
After drop — Train: (590540, 425)  Test: (506691, 433)


## 4. Add Derived Features (via src.features)

In [5]:
# Add log_amount, hour_of_day, amount_zscore
# Training mode: stats computed from training set
train_engineered = engineer_features(train, amount_mean=None, amount_std=None)

# Inference mode: use training stats to avoid leakage
train_amount_mean = float(train['TransactionAmt'].mean())
train_amount_std  = float(train['TransactionAmt'].std())
test_engineered   = engineer_features(test, amount_mean=train_amount_mean,
                                      amount_std=train_amount_std)

print('New columns added:', ['log_amount', 'hour_of_day', 'amount_zscore'])
print(train_engineered[['TransactionAmt','log_amount','hour_of_day','amount_zscore']].head(3))

New columns added: ['log_amount', 'hour_of_day', 'amount_zscore']
   TransactionAmt  log_amount  hour_of_day  amount_zscore
0            68.5    4.241327            0      -0.278167
1            29.0    3.401197            0      -0.443327
2            59.0    4.094345            0      -0.317889


## 5. Velocity Features (card1 rolling window)

In [6]:
# Backward-looking velocity per card1: 1-day and 7-day windows
# Combined train+test for consistency, split back after

target    = train_engineered['isFraud'].copy()
train_ids = train_engineered['TransactionID'].copy()
test_ids  = test_engineered['TransactionID'].copy()

train_feat = train_engineered.drop(columns=['isFraud', 'TransactionID'])
test_feat  = test_engineered.drop(columns=['TransactionID'])

n_train = len(train_feat)
combined = pd.concat([
    train_feat.assign(TransactionID=train_ids.values, isFraud=target.values, _split='train'),
    test_feat.assign(TransactionID=test_ids.values,   isFraud=np.nan,        _split='test'),
], ignore_index=True).sort_values('TransactionDT').reset_index(drop=True)

combined['_dt'] = pd.to_datetime(combined['TransactionDT'], unit='s', origin='unix')
combined = combined.set_index('_dt').sort_index()

def rolling_card1(df, window):
    return (df.groupby('card1')['TransactionAmt']
              .transform(lambda s: s.shift(1).rolling(window, min_periods=1).sum()))

combined['vel_1d_count']  = (combined.groupby('card1')['TransactionAmt']
    .transform(lambda s: s.shift(1).rolling('1D', min_periods=1).count()))
combined['vel_1d_amt']    = (combined.groupby('card1')['TransactionAmt']
    .transform(lambda s: s.shift(1).rolling('1D', min_periods=1).sum()))
combined['vel_7d_count']  = (combined.groupby('card1')['TransactionAmt']
    .transform(lambda s: s.shift(1).rolling('7D', min_periods=1).count()))
combined['vel_7d_amt']    = (combined.groupby('card1')['TransactionAmt']
    .transform(lambda s: s.shift(1).rolling('7D', min_periods=1).sum()))
combined['vel_7d_merch']  = (combined.groupby('card1')['MerchantID']
    .transform(lambda s: s.shift(1).rolling('7D', min_periods=1).nunique())
    if 'MerchantID' in combined.columns else 0)
combined['vel_7d_dev']    = (combined['vel_7d_amt'] /
    combined.groupby('card1')['vel_7d_count'].transform('mean').replace(0, np.nan))

combined = combined.reset_index(drop=True)
print('Velocity features added. Sample:')
print(combined[['vel_1d_count','vel_1d_amt','vel_7d_count','vel_7d_amt']].head(3))

Velocity features added. Sample:
   vel_1d_count  vel_1d_amt  vel_7d_count  vel_7d_amt
0           0.0         NaN           0.0         NaN
1           0.0         NaN           0.0         NaN
2           0.0         NaN           0.0         NaN


## 6. Categorical Encoding

In [7]:
# Split back before encoding to fit on train only
cat_cols = combined.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c not in ['_split']]
print(f'Encoding {len(cat_cols)} categorical columns')

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    combined[col] = le.fit_transform(combined[col].astype(str))
    encoders[col] = le

print('Encoding done')

Encoding 44 categorical columns
Encoding done


## 7. Train/Validation Split

In [8]:
train_final = combined[combined['_split'] == 'train'].copy()
test_final  = combined[combined['_split'] == 'test'].copy()
train_final = train_final.sort_values('TransactionDT').reset_index(drop=True)

drop_meta = ['_split', 'isFraud', 'TransactionID']
feature_cols = [c for c in train_final.columns if c not in drop_meta]

X = train_final[feature_cols]
y = train_final['isFraud'].astype(int)
X_test_final = test_final[feature_cols]

# Time-based 80/20 split (preserve temporal order — no shuffle)
split_idx = int(len(X) * 0.80)
X_tr, X_val = X.iloc[:split_idx], X.iloc[split_idx:]
y_tr, y_val = y.iloc[:split_idx], y.iloc[split_idx:]

print(f'X_tr: {X_tr.shape}  X_val: {X_val.shape}')
print(f'Train fraud: {y_tr.mean():.4f}  Val fraud: {y_val.mean():.4f}')

X_tr: (472432, 470)  X_val: (118108, 470)
Train fraud: 0.0351  Val fraud: 0.0344


## 8. SMOTE+Tomek Resampling (training set only)

> **Important:** SMOTE is applied **only to X_tr** to prevent leakage into validation/test.

In [9]:
from sklearn.impute import SimpleImputer

print(f'Before SMOTE — X_tr: {X_tr.shape}  fraud: {y_tr.sum():,} ({y_tr.mean()*100:.2f}%)')

imputer = SimpleImputer(strategy='median', keep_empty_features=True)
X_tr_imp  = pd.DataFrame(imputer.fit_transform(X_tr),   columns=X_tr.columns)
X_val_imp = pd.DataFrame(imputer.transform(X_val),      columns=X_val.columns)

smt = SMOTETomek(random_state=SEED)
X_tr_res, y_tr_res = smt.fit_resample(X_tr_imp, y_tr)

print(f'After  SMOTE — X_tr_res: {X_tr_res.shape}  fraud: {y_tr_res.sum():,} ({y_tr_res.mean()*100:.2f}%)')

Before SMOTE — X_tr: (472432, 470)  fraud: 16,599 (3.51%)
After  SMOTE — X_tr_res: (904270, 470)  fraud: 452,135 (50.00%)


## 9. Save Processed Data

In [10]:
import os
os.makedirs('../data/processed', exist_ok=True)

# Save processed splits for use in training notebook
X_tr.to_parquet('../data/processed/X_tr_raw.parquet')
X_val.to_parquet('../data/processed/X_val_raw.parquet')
pd.DataFrame(X_tr_res, columns=X_tr.columns).to_parquet('../data/processed/X_tr_smt.parquet')
pd.DataFrame(X_val_imp, columns=X_val.columns).to_parquet('../data/processed/X_val_imp.parquet')
y_tr.to_frame().to_parquet('../data/processed/y_tr.parquet')
y_val.to_frame().to_parquet('../data/processed/y_val.parquet')
pd.Series(y_tr_res, name='isFraud').to_frame().to_parquet('../data/processed/y_tr_smt.parquet')
X_test_final.to_parquet('../data/processed/X_test.parquet')
test_ids.to_frame().to_parquet('../data/processed/test_ids.parquet')

# Save training stats and encoders for inference
joblib.dump({'mean': train_amount_mean, 'std': train_amount_std},
            '../models/amount_stats.pkl')
joblib.dump(encoders, '../models/label_encoders.pkl')
joblib.dump(imputer,  '../models/imputer.pkl')
joblib.dump(feature_cols, '../models/feature_cols.pkl')

print('Saved all processed splits and artifacts')
print(f'feature_cols: {len(feature_cols)}')

Saved all processed splits and artifacts
feature_cols: 470


## Summary

| Step | Detail |
|------|--------|
| Dropped | Columns >95% missing |
| Added | `log_amount`, `hour_of_day`, `amount_zscore` (via `src.features`) |
| Added | 6 card1 velocity features (1-day and 7-day rolling) |
| Encoded | All object columns via LabelEncoder (fit on combined train+test) |
| Split | 80/20 time-based (no shuffle) |
| SMOTE | Applied to training set only — avoids leakage |
| Saved | Parquet splits in `data/processed/`, artifacts in `models/` |